# nano-2: `fixed-mn` baseline

Direct reproduction of the original notebook config. Random Beta(0.5, 0.5)-sampled m_n masks, no learn-assign, no annealing.

Cloud RunPod 3090 currently produces slider-best ≈ 2.27 with this config. Original Mac CPU notebook produced ≈ 1.98. **Run this in Colab to see which side T4 fp32/bf16 lands on.**

ETA: ~15-25 min on Colab T4 (uses bf16 autocast).

In [ ]:
# Setup: clone repo + install deps
!git clone -q https://github.com/iamtrask/abcGPT.git
%cd abcGPT
!pip install -q numpy

# Prep data (shake + tinystories char-level)
!python3 data/shakespeare_tinystories_char/prepare.py

In [ ]:
# Verify torch + GPU
import torch
print(f'torch:        {torch.__version__}')
print(f'cuda:         {torch.version.cuda}')
print(f'gpu:          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'bf16 support: {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}')

In [ ]:
# Train fixed-mn for 10k iters with default config.
# Same args as cloud's `fixed-mn-replicate-seeded` variant.
!python3 experiments/nano-2/train.py \
    --variant-name colab-fixed-mn \
    --variant fixed-mn \
    --n-iters 10000 \
    --span 1.0 \
    --seed 1337 \
    --eval-interval 500 \
    --log-interval 250 \
    --device cuda \
    --amp-dtype bfloat16

In [ ]:
# Parse log and plot the alpha-curve
import json
from pathlib import Path
import matplotlib.pyplot as plt

log_path = Path('experiments/nano-2/results/colab-fixed-mn/log.jsonl')
records = [json.loads(line) for line in open(log_path)]
evals = [r for r in records if r['type'] == 'eval']
alpha_curve = [r for r in records if r['type'] == 'alpha_curve']

# Best eval by sum-loss
sums = [(r['iter'], r['val_loss_matrix']['shake@a=1.0'] + r['val_loss_matrix']['ts@a=0.0']) for r in evals]
best_iter, best_sum = min(sums, key=lambda x: x[1])
final_iter, final_sum = sums[-1]
print(f'best endpoint-sum: {best_sum:.4f} at iter {best_iter}')
print(f'final endpoint-sum: {final_sum:.4f} at iter {final_iter}')

if alpha_curve:
    pts = alpha_curve[-1]['points']
    alphas = [p['alpha'] for p in pts]
    sh = [p['shake_val'] for p in pts]
    ts = [p['ts_val'] for p in pts]
    sh_min = min(sh); ts_min = min(ts)
    print(f'slider-best (min_a sh + min_a ts): {sh_min + ts_min:.4f}')
    print(f'shake min at alpha={alphas[sh.index(sh_min)]:.2f}: {sh_min:.4f}')
    print(f'ts min at alpha={alphas[ts.index(ts_min)]:.2f}: {ts_min:.4f}')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(alphas, sh, 'o-')
    axes[0].set_xlabel('alpha (1.0 = shake mode)')
    axes[0].set_ylabel('val loss on tinyshakespeare')
    axes[0].set_title('shake val')
    axes[0].grid(alpha=0.3)
    axes[1].plot(alphas, ts, 'o-')
    axes[1].set_xlabel('alpha (1.0 = shake mode)')
    axes[1].set_ylabel('val loss on TinyStories')
    axes[1].set_title('tinystories val')
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('(no alpha curve recorded)')